# Modules => Command-Line Arguments

Programs often read options from the command line, such as `tool.py data.csv --limit 5 --verbose`.

| Tool | Purpose |
|---|---|
| `sys.argv` | The raw list of command-line strings. `argv[0]` is the script name |
| `argparse.ArgumentParser` | Builds a parser with help and error messages |
| `add_argument()` | Declares one argument |
| `parse_args()` | Reads and validates the arguments |
| Positional argument | Required by position: `add_argument("file")` |
| Optional argument | Starts with `-` or `--`: `add_argument("--limit")` |
| `type=` | Converts the text (`int`, `float`, ...) |
| `default=` | Value when the argument is missing |
| `choices=` | Restricts the allowed values |
| `action="store_true"` | A flag with no value (`--verbose`) |
| `nargs=` | How many values to accept |
| `add_subparsers()` | Sub-commands such as `git commit` |

---

## `sys.argv`

```python
import sys
print(sys.argv)     # ['tool.py', 'data.csv', '--limit', '5']
```

Every item is a **string**. You must parse and validate them yourself. For anything beyond one or two arguments, use `argparse`.

---

## `argparse` Basics

```python
import argparse

parser = argparse.ArgumentParser(prog="tool", description="Process a file.")
parser.add_argument("file")
parser.add_argument("--limit", type=int, default=10)
parser.add_argument("-v", "--verbose", action="store_true")

args = parser.parse_args()
print(args.file, args.limit, args.verbose)
```

* `parse_args()` reads `sys.argv[1:]` when called without arguments.
* Dashes in option names become underscores: `--max-size` is `args.max_size`.
* `-h` and `--help` are added automatically.
* Invalid input prints a usage message and exits with code `2`.

---

## `add_argument()` Options

| Option | Example | Meaning |
|---|---|---|
| `type` | `type=int` | Convert to `int` |
| `default` | `default=10` | Value when missing |
| `required` | `required=True` | Makes an optional argument mandatory |
| `choices` | `choices=["a", "b"]` | Only these values |
| `help` | `help="Max rows"` | Text shown in `--help` |
| `metavar` | `metavar="N"` | Name shown in the usage line |
| `action` | `"store_true"` | Flag: `True` when present |
| `action` | `"count"` | Counts repeats: `-vvv` gives `3` |
| `nargs` | `nargs="+"` | One or more values |
| `nargs` | `nargs="?"` | Zero or one value |

---

## Sub-Commands

```python
sub = parser.add_subparsers(dest="command")
add = sub.add_parser("add")
add.add_argument("name")
```

Each sub-command has its own arguments, like `git add` and `git commit`.

---

## In a Notebook

A notebook has no command line. Pass a list to `parse_args()` to test:

```python
args = parser.parse_args(["data.csv", "--limit", "5"])
```

## Source

https://docs.python.org/3/library/argparse.html

https://docs.python.org/3/tutorial/stdlib.html#command-line-arguments

In [ ]:
import argparse
import contextlib
import io
import sys

# sys.argv: the raw list of strings (in a notebook it holds the kernel's own arguments)
print(type(sys.argv).__name__, all(isinstance(item, str) for item in sys.argv))

# Build a parser
parser = argparse.ArgumentParser(prog="tool", description="Process a file.")
parser.add_argument("file", help="input file")
parser.add_argument("--limit", type=int, default=10, help="max rows")
parser.add_argument("--mode", choices=["fast", "safe"], default="safe")
parser.add_argument("-v", "--verbose", action="store_true")
parser.add_argument("-c", "--count", action="count", default=0)
parser.add_argument("--tags", nargs="+", default=[])

# In a notebook, pass a list instead of reading the real command line
args = parser.parse_args(["data.csv", "--limit", "5", "-v", "-ccc", "--tags", "a", "b"])
print(args)
print(args.file, args.limit, args.verbose, args.count, args.tags)

# Defaults when nothing is given
print(parser.parse_args(["data.csv"]))

# The help text is generated for you
print(parser.format_usage())

# Errors print a usage message and exit with code 2
for bad in (["data.csv", "--limit", "abc"], ["data.csv", "--mode", "turbo"], []):
    with contextlib.redirect_stderr(io.StringIO()):
        try:
            parser.parse_args(bad)
        except SystemExit as error:
            print("exit code", error.code, "for", bad)

# Sub-commands
cli = argparse.ArgumentParser(prog="todo")
sub = cli.add_subparsers(dest="command", required=True)
add = sub.add_parser("add")
add.add_argument("title")
done = sub.add_parser("done")
done.add_argument("number", type=int)

print(cli.parse_args(["add", "buy milk"]))
print(cli.parse_args(["done", "3"]))